# RSNA Knee — fine-tuned ResNet18 (3 planes, attention pooling over slices)
GPU必須。Internet ONでImageNet重みを取得。画像は224px×16スライスをディスクにキャッシュし、エンドツーエンドで学習する。

In [ ]:
import os, sys, subprocess, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'  # joblibのワーカープロセスにも効かせる
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
_name = 'rsna-knee-abnormality-detection'
_cands = [Path('/kaggle/input/competitions')/_name, Path('/kaggle/input')/_name]
COMP = next((d for d in _cands if (d/'train.csv').exists()), None)
assert COMP, f"train.csv が見つからない。/kaggle/input: {[p.name for p in Path('/kaggle/input').iterdir()]}"
print('COMP =', COMP)
REPO = Path('/kaggle/temp/repo') if Path('/kaggle/temp').exists() else Path('/tmp/repo')   # /kaggle/working に置くと実行の出力に含まれてしまう
SRC = REPO/'rsna-knee-abnormality-detection'/'src'
if not SRC.exists():  # internet ON のときだけ。OFFなら src を Dataset で追加して SRC を変更
    subprocess.run(['git','clone','--depth','1','-b','main',
                    'https://github.com/hidenori24/kaggle_ipynb', str(REPO)], check=True)
sys.path.insert(0, str(SRC))
from pseudo_labels import LABELS
from multilingual_labels import add_pseudo_labels_ml
import finetune as ft, torch
print('commit:', subprocess.run(['git','-C',str(SRC),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

In [ ]:
train = pd.read_csv(COMP/'train.csv'); tseries = pd.read_csv(COMP/'train_series.csv')
test = pd.read_csv(COMP/'test.csv');   teseries = pd.read_csv(COMP/'test_series.csv')
print(train.shape, tseries.shape, test.shape)
train['has_true'] = train['ACL'].notna()   # 正解ラベルあり(疑似ラベルではない)
print('true-labeled:', int(train.has_true.sum()))
train = add_pseudo_labels_ml(train)        # 英語+トルコ語+ギリシャ語+ブルガリア語+スペイン語+ドイツ語+フランス語+オランダ語+クロアチア語のキーワード規則
print(train.lang.value_counts().to_dict())
# ---- ラベルの出どころ: 'llm' = 06_llm_labels が作ったソフトラベル(Dataset rsna-knee-labels)。無ければ 'rules'(キーワード規則)
LABEL_SOURCE = 'llm'
def find_dir(name):
    for base in [Path('/kaggle/input'), Path('/kaggle/input/datasets')]:
        if not base.exists(): continue
        for d in base.iterdir():
            if d.name == name: return d
            if d.is_dir() and d.name not in ('competitions', 'datasets'):
                for s in d.iterdir():
                    if s.name == name: return s
    return None
train['llm_ok'] = False
_d = find_dir('rsna-knee-labels') if LABEL_SOURCE == 'llm' else None
if _d is not None and (_d / 'llm_labels.csv').exists():
    L = pd.read_csv(_d / 'llm_labels.csv').drop_duplicates('StudyInstanceUID').set_index('StudyInstanceUID')
    soft = L.reindex(train.StudyInstanceUID)[LABELS].values
    ok = ~np.isnan(soft).any(1) & ~train.has_true.values                  # 正解ラベル付きは正解のまま
    for j, c in enumerate(LABELS):
        train.loc[ok, c] = soft[ok, j]
    train['llm_ok'] = ok
    print('LLM soft labels used for', int(ok.sum()), 'studies; mean per label:', dict(zip(LABELS, soft[ok].mean(0).round(2))))
else:
    LABEL_SOURCE = 'rules'
    print('LLM labels not found -> keyword rules')
# 言語ごとの疑似陽性率(本文は出さない)。英語と桁が大きく違う言語は規則が弱い疑い
print(train[~train.has_true].groupby('lang')[LABELS].mean().round(3).T)
# 診断: 規則の弱い言語(bg, tr)と未対応('other')の頻出語。本文は出さず、語の文書頻度だけを出す
import re
from collections import Counter
def top_words(mask, n):
    docs = train.loc[mask, 'Report'].fillna('').astype(str).str.lower()
    c = Counter(w for t in docs for w in set(re.findall(r'[^\W\d_]{4,}', t)))
    return c.most_common(n)
for lg, n in (('bg', 120), ('tr', 120), ('other', 150)):
    print(f'top words [{lg}] ({int((train.lang == lg).sum())} reports):', top_words(train.lang == lg, n))

# 診断: キーワード規則が、正解ラベル付き58件にどれだけ合っているか(規則ラベルは正解を見ずに作り直す)
from multilingual_labels import label_report_lang
tv = train[train.has_true]
R = np.array([[label_report_lang(r, l)[c] for c in LABELS] for r, l in zip(tv.Report.fillna(''), tv.lang)])
T = tv[LABELS].values.astype(int)
tp, fp, fn = (R & T).sum(0), (R & (1 - T)).sum(0), ((1 - R) & T).sum(0)
print('rule labels vs TRUE labels (n=%d, languages: %s)' % (len(tv), tv.lang.value_counts().to_dict()))
print(pd.DataFrame({'true_pos': T.sum(0), 'TP': tp, 'FP': fp, 'FN': fn}, index=LABELS))
print('micro precision %.3f  recall %.3f' % (tp.sum() / max(tp.sum() + fp.sum(), 1), tp.sum() / max(tp.sum() + fn.sum(), 1)))


In [ ]:
SIZE, K, EPOCHS, BS = 224, 16, 4, 8
CENTER = 1.0       # 0.7 にすると各seriesの中央70%から16枚を取る(端のスライスは膝が写っていない)
USE_META = False   # True にするとDICOMヘッダー(スキャナー・スライス厚・解像度など)もモデルに入力する
CACHE = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path('/tmp')   # /kaggle/working に置くと出力に含まれてしまう
USE_LANGS = ['en', 'tr', 'el', 'bg', 'es', 'de', 'fr', 'nl', 'hr']   # 使う言語。英語のみで比較したいときは ['en']
sel = train.has_true | (train.llm_ok if LABEL_SOURCE == 'llm' else train.lang.isin(USE_LANGS))   # LLMラベルは言語を問わない
use = train[sel].reset_index(drop=True)   # 疑似ラベル + 正解ラベル付き
print('label source:', LABEL_SOURCE)
print(use.lang.value_counts().to_dict())
print('studies:', len(use), '| true-labeled:', int(use.has_true.sum()))
X, M, META = ft.cache_volumes(use, tseries, COMP/'train_series', SIZE, K, CACHE, 'train', center=CENTER)
META_ARG = META if USE_META else None
from dicom_meta import _MFR
print('manufacturers (siemens/ge/philips/...):', dict(zip(_MFR, META[:, 3 * ft.N_NUM:].sum(0).astype(int))),
      '| studies with header meta:', int((META[:, :ft.N_NUM] != 0).any(1).sum()))
print('planes available (Sag/Cor/Ax):', M.sum(0))
Y = use[LABELS].values.astype(np.float32)

In [ ]:
N_SEEDS = 3                                   # 乱数と検証分割を変えて3モデル。推論時に予測を平均する
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
OUT = Path('model_out'); OUT.mkdir(exist_ok=True)
true_idx = np.where(use.has_true.values)[0]                       # 58件は評価専用(正解ラベル)
pseudo_all = np.where(~use.has_true.values)[0]
nets, true_preds = [], []
for seed in range(N_SEEDS):
    rng = np.random.RandomState(seed); pseudo = pseudo_all.copy(); rng.shuffle(pseudo)
    n_va = int(0.1 * len(pseudo))
    va_pseudo, tr_idx = pseudo[:n_va], pseudo[n_va:]                # シードごとに別の10%を検証に使う
    torch.manual_seed(seed); np.random.seed(seed)
    print(f'=== seed {seed}: train {len(tr_idx)} val-pseudo {len(va_pseudo)}')
    net, hist = ft.fit(X, M, Y, tr_idx, {'pseudo': va_pseudo, 'true': true_idx}, dev, epochs=EPOCHS, bs=BS, META=META_ARG)
    print('best:', max(hist, key=lambda h: h.get('pseudo', -1)))
    torch.save({'state': net.state_dict(), 'size': SIZE, 'k': K, 'center': CENTER, 'meta': USE_META, 'labels': LABELS}, OUT/f'ft_model_s{seed}.pt')  # 各モデルをすぐ保存
    true_preds.append(ft.predict(net, X, M, true_idx, dev, META=META_ARG))
    nets.append(net.cpu()); torch.cuda.empty_cache()

In [ ]:
for i, P in enumerate(true_preds):
    print(f'seed {i}: macro AUC vs TRUE labels {ft.macro_auc(Y[true_idx], P):.4f}')
print(f'ensemble ({N_SEEDS} models): macro AUC vs TRUE labels {ft.macro_auc(Y[true_idx], np.mean(true_preds, axis=0)):.4f}')
print(ft.per_label_auc(Y[true_idx], np.mean(true_preds, axis=0), LABELS).round(3))   # どの所見が弱いか
